In [ ]:
import sys
from pathlib import Path

root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()),
    None,
)
if root and str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from py_entry.strategy_hub import (
    build_strategy_runtime,
    get_strategy_refs,
)


strategy_refs = get_strategy_refs()
print("可用策略:", [i for i in strategy_refs if i.startswith("search:")])

STRATEGY = strategy_refs[0]
STRATEGY = "search:macd_plain.macd_plain"
RUN_SYMBOL = "DOGE/USDT"
print("当前策略:", STRATEGY)
print("运行品种:", RUN_SYMBOL)

cfg, stage_cfgs, bt = build_strategy_runtime(
    STRATEGY,
    run_symbol=RUN_SYMBOL,
)

In [ ]:
# WF 预检（显式调用一次，失败直接抛错）
wf_precheck = bt.validate_wf_indicator_readiness(stage_cfgs["wf_cfg"])
print(
    "WF预检通过:",
    {
        "base_data_key": wf_precheck["base_data_key"],
        "indicator_warmup_bars_base": wf_precheck["indicator_warmup_bars_base"],
        "effective_transition_bars": wf_precheck["effective_transition_bars"],
    },
)

In [ ]:
# 阶段1：回测
backtest_result = bt.run()
backtest_result.print_report()

In [ ]:
from py_entry.io import DisplayConfig, DashboardOverride
from py_entry.runner import FormatResultsConfig

config = DisplayConfig(
    embed_data=False,
    width="100%",
    aspect_ratio="16/9",
    override=DashboardOverride(
        show=["0,0,0,1"],
        showInLegend=["0,0,0,1"],
        showRiskLegend="1,1,1,1",
        showLegendInAll=True,
    ).to_dict(),
)

backtest_result = backtest_result.format_for_export(
    FormatResultsConfig(dataframe_format="parquet", indicator_layout=cfg.chart_layout)
)
backtest_result.display(config=config)

In [ ]:
# 阶段2：优化
opt_result = bt.optimize(stage_cfgs["opt_cfg"])
opt_result.print_report()

In [ ]:
# 阶段3：参数抖动
sens_result = bt.sensitivity(stage_cfgs["sens_cfg"])
sens_result.print_report()

In [ ]:
# 阶段4：向前测试
wf_result = bt.walk_forward(stage_cfgs["wf_cfg"])
wf_result.print_report()

In [ ]:
wf_result = wf_result.format_for_export(
    FormatResultsConfig(dataframe_format="parquet", indicator_layout=cfg.chart_layout)
)
wf_result.display(config=config)